# Build Your Own AI Agent Harness for Finance
<font color='#2563eb'><b>FDE Playbook Workshop Notebook</b></font>

---

## What is an AI Agent?

Most people think an AI Agent is simply a smarter Large Language Model (LLM). But in production engineering:

$$\mathbf{Agent = LLM + Harness}$$

$$\mathbf{Harness = Tools + Memory + Logic / Orchestration}$$

-----------------------------------
### Why This Matters for Forward Deployed Engineers (FDEs)
Foundational models (Gemini, Claude, GPT) are readily available commodities. **The true competitive advantage of an AI-focused FDE is engineering the Harness.**

In this hands-on workshop, we will build a complete finance agent harness step by step—progressing from a LLM to an institutional multi-agent research team and cloud-managed agents.

---

### The Harness Architecture at a Glance

```
+-----------------------------------------------------------------------------------+
|                                  AI AGENT HARNESS                                 |
|                                                                                   |
|  [ User Request ]                                                                 |
|         |                                                                         |
|         v                                                                         |
|  +--------------------+        +---------------------+        +----------------+  |
|  |       MEMORY       | <----> |      THE BRAIN      | <----> |     TOOLS      |  |
|  | - Investment Style |        |     (LLM / Gemini)  |        | - DuckDuckGo   |  |
|  | - Risk Tolerance   |        +---------------------+        | - YahooFinance |  |
|  | - Past Context     |                   |                   +----------------+  |
|  +--------------------+                   v                                       |
|                                +---------------------+                            |
|                                | LOGIC / ORCHESTRATION|                           |
|                                | - Financial Analysis|                            |
|                                | - Scorecard Tables  |                            |
|                                | - Multi-Agent Team  |                            |
|                                +---------------------+                            |
|                                           |                                       |
|                                           v                                       |
|                            [ Institutional Investment Memo ]                      |
+-----------------------------------------------------------------------------------+
```


## 1. Setup

We install **Agno**—a modern, lightweight framework designed for agentic harnesses—along with domain tools for financial market data (`yfinance`), live web access (`duckduckgo-search`), and persistent SQLite storage. We also include Anthropic for cloud-managed agents.


In [ ]:
!pip install -qU "agno[sqlite]" yfinance duckduckgo-search google-genai anthropic sqlalchemy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 35.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take 

### API Key Configuration

You will need a free Gemini API key from [Google AI Studio](https://aistudio.google.com).
* In Google Colab, you can store your key in the **Secrets** tab (key icon on the left) under `GOOGLE_API_KEY`.
* Or simply paste your API key directly below.


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## 2. The Simple LLM Call (The Brain Without a Harness)

Let's begin by testing a simple LLM call without any harness components (no tools, no memory, no external data).

We ask it a simple real-world financial question: **What is the current stock price and latest market news for NVIDIA (NVDA) right now?**


In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

# A Simple LLM: No tools, no memory, no harness
simple_llm = Agent(
    model=Gemini(id="gemini-3.8-flash"),
    markdown=True,
)

# Ask for real-time market data
simple_llm.print_response(
    "What is the current stock price of NVIDIA (NVDA) right now, and what are the top market headlines today?",
    stream=True
)


Output()


> Notice the result! The above LLM call cannot give you the current stock price or today's news. It either admits that its knowledge cutoff date prevents it from accessing real-time information, or it provides outdated historical data.


## 3. Harness Pillar 1 — Tools (Giving the Agent Hands & Eyes)

The first component of an Agent Harness is **Tools**. Tools allow the LLM to reach out to the external world to fetch live data, query search engines, and invoke APIs.

We will equip our agent with two levels of tools:
1. **Live Web Search (`DuckDuckGoTools`)**: To capture real-time headlines, industry sentiment, and breaking news.
2. **Domain Financial Market Tools (`YFinanceTools`)**: To retrieve live ticker prices, analyst consensus, P/E multiples, and fundamentals.


### 3.1 Adding Live Web Search
Let's equip our agent with live web search so it can look up breaking market developments across the internet.


In [ ]:
!pip install -q ddgs agno

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.websearch import WebSearchTools

agent = Agent(
    model=Gemini(id="gemini-3.8-flash"),
    tools=[WebSearchTools(backend="auto")],
    instructions=[
        "Search for news published during given time period from trusted sources only.",
        "Never assume the year. Verify dates from search results."
    ],
    markdown=True,
    add_datetime_to_context=True,
)

agent.print_response(
    "Find and summarize the latest NVIDIA (NVDA) news and Blackwell AI chip updates from the last week. Include dates and source URLs.",
    stream=True,
)

ERROR    GOOGLE_API_KEY not set. Please set the GOOGLE_API_KEY environment variable.

ERROR    Unknown error from Gemini API: No API key was provided. Please pass a valid API key. Learn how to create  
         an API key at https://ai.google.dev/gemini-api/docs/api-key.

ERROR    Error in Agent run: No API key was provided. Please pass a valid API key. Learn how to create an API key  
         at https://ai.google.dev/gemini-api/docs/api-key.

### 3.2 Adding Domain Financial Tools (`YFinanceTools`)
Web search is great for qualitative news, but financial analysis requires **hard quantitative numbers**: exact stock prices, market cap, 52-week ranges, and Wall Street analyst targets.

With Agno, we can plug in `YFinanceTools` directly. Notice how clean the code is: no custom API wrappers, no scraping loops, and no messy parsing code!


In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.yfinance import YFinanceTools

finance_tool_agent = Agent(
    model=Gemini(id="gemini-3.8-flash"),
    tools=[YFinanceTools(
        enable_stock_price=True,
        enable_analyst_recommendations=True,
        enable_company_info=True,
        enable_company_news=True,
    )],
instructions=[
    """Use simple language and Markdown under 200 words.
    Include a heading and bullets for current price, 52-week range, and analyst recommendations.
    Explain key figures briefly. End with 2-3 takeaways.
    Never invent data; mention unavailable information."""
],
    markdown=True,
)

finance_tool_agent.print_response(
    "Explain NVIDIA (NVDA) stock's current price, 52-week range, "
    "and Wall Street analyst recommendations to a beginner. "
    "What is happening, why does it matter, and what should a general reader understand?",
    stream=True,
)

Output()

> ### Notice the Difference!
> With just a few lines of clean code, our agent:
> 1. Decided which financial tool to call.
> 2. Queried Yahoo Finance for live market data.
> 3. Formatted the numerical data into clean markdown tables with streaming output.
>
> We didn't write a single boilerplate loop, regex parser, or print statement. The harness handled everything out of the box.


## 4. Harness Pillar 2 — Memory (Remembering User Preferences)

The second pillar of an Agent Harness is Memory.

Without memory, an agent forgets previous conversations and starts from scratch each time. But investment advice depends on a person's goals and preferences.

- Someone who is comfortable with risk may invest differently from someone who wants to protect their savings.
- If a user says, "I want to invest in technology stocks for the next 3–5 years," the agent should remember this for future conversations.

Let's add memory to our agent using `SqliteDb` and `add_history_to_context=True` so it can use previous conversations to give more relevant answers.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.yfinance import YFinanceTools
from agno.db.sqlite import SqliteDb

# Store conversation history and user memories
advisor_db = SqliteDb(db_file="tmp/finance_agent.db")

In [ ]:
personalized_advisor = Agent(
    model=Gemini(id="gemini-3.8-flash"),
    tools=[
        YFinanceTools(
            enable_stock_price=True,
            enable_analyst_recommendations=True,
            enable_company_info=True,
        )
    ],
    db=advisor_db,
    add_history_to_context=True,
    num_history_runs=3,
    enable_agentic_memory=True,
    add_memories_to_context=True,
instructions=[
    """Explain stock investing in simple, everyday language under 150 words.
    Use actual stock data and specific numbers to explain what is happening and why it matters to the user.
    Avoid complex financial terms, generic advice, and unsupported predictions. Mention risks and end with one clear takeaway."""
],
    markdown=True,
)

###Turn 1
I want to invest in stocks for the next 3–5 years. I can handle some ups and downs, but I want to understand the risks before investing. I am interested in technology and AI companies. Please remember my preferences for future advice.

In [ ]:
personalized_advisor.print_response(
    "I want to grow my savings by investing over the next 3–5 years. "
    "I can accept some ups and downs but want to understand the risks. "
    "I am especially interested in AI and technology companies. "
    "Please remember these preferences for future guidance.",
    user_id="satvik",
    stream=True,
)

Output()

Now, in **Turn 2**, notice that we ask about NVDA **without restating our profile**.

Watch how the agent automatically recalls our aggressive risk tolerance, 3-5 year time horizon, and AI infrastructure focus from its memory:


In [ ]:
personalized_advisor.print_response(
    "What 2-3 stocks should I research based on my investment preferences?",
    user_id="satvik",
    stream=True,
)

Output()

In [ ]:
from pprint import pprint

memories = personalized_advisor.get_user_memories(user_id="satvik")
pprint(memories, width=80)

[UserMemory(memory='User has an investment horizon of 3–5 years, a moderate '
                   'risk tolerance comfortable with some market fluctuations, '
                   'and a goal to grow savings.',
            memory_id='37242108-2207-429a-bd58-5275634a2e1f',
            topics=['finances', 'investment', 'goals'],
            user_id='satvik',
            input="Save user's investment preferences: Horizon of 3–5 years, "
                  'moderate risk tolerance (comfortable with some market '
                  'fluctuations), goal to grow savings, and special interest '
                  'in AI and technology companies.',
            created_at=1791624356,
            updated_at=1791624356,
            feedback=None,
            agent_id=None,
            team_id=None),
 UserMemory(memory='User has a special investment interest in AI and '
                   'technology companies.',
            memory_id='cdac8325-a516-4760-9640-863e95c18172',
            topics=['finances'

> ### Why Memory Makes the Advisor Better
>
> The agent didn't give general advice. It used the user's risk comfort, investment time frame, and interest in AI and technology to make its response more relevant.
>
> Memory helps the AI give more personalized guidance based on previous conversations.

## 5. Harness Pillar 3 — Logic & Analysis

Tools provide data, and memory remembers user preferences. Logic helps the agent make sense of the information.

Without clear instructions, the agent might miss important risks or provide numbers without context.

We guide the agent to check:

1. Price: Is the stock worth its price?
2. Strengths: What makes the company stand out?
3. Risks: What could go wrong?
4. Pros & Cons: Compare potential benefits and risks.

We use `ReasoningTools` to support structured analysis and clearer answers.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.reasoning import ReasoningTools
from agno.tools.yfinance import YFinanceTools

# Stock Analyst with Structured Reasoning
stock_analyst = Agent(
    model=Gemini(id="gemini-3.8-flash"),
    tools=[
        ReasoningTools(add_instructions=True),
        YFinanceTools(
            enable_stock_price=True,
            enable_analyst_recommendations=True,
            enable_company_info=True,
            enable_company_news=True,
        ),
    ],
instructions=[
    """Analyze stocks in simple language with clear headings and bullets.
    Include current price, 52-week range, analyst views, and 2–3 useful figures with brief explanations.
    Cover growth opportunities and the top 3 risks in one sentence each; compare figures where useful.
    End with a balanced bottom line on the opportunity and key risks.
    Aim for 150–200 words. Avoid jargon, repetition, long paragraphs, and unnecessary tables.
    Use current, reliable tool data; separate facts from estimates. Never invent figures or guarantee returns."""
],
    markdown=True,
)

stock_analyst.print_response(
    "Give me a simple summary of NVIDIA (NVDA), including its current price, growth potential, and key risks.",
    stream=True,
)

Output()

> ### Harness Pillar Check:
> * $\checkmark$ **Tools:** Fetched live numbers from Yahoo Finance.
> * $\checkmark$ **Memory:** Preserves user and session context.
> * $\checkmark$ **Logic / Orchestration:** Guided the model through rigorous multi-step financial reasoning and scorecard synthesis.
>
> Now let's see what happens when we scale from **one agent** to a **multi-agent team**!


## 6. Level Up — Multi-Agent Harness (A Financial Research Team)

In real financial research, different people focus on different tasks:

- News Researcher: Tracks company news, market changes, and new regulations.
- Financial Analyst: Checks stock prices, company performance, and analyst opinions.
- Research Lead: Combines their findings into one clear summary.

With Agno, we can build a team of AI agents using `Team`, where each agent handles a specific task and the leader brings everything together.

In [ ]:
%pip install -U ddgs

In [ ]:

from agno.agent import Agent
from agno.team import Team
from agno.models.google import Gemini
from agno.tools.yfinance import YFinanceTools

model = Gemini(id="gemini-3.8-flash")

# 1. Company News Researcher
news_researcher = Agent(
    name="Company News Researcher",
    role="Find and summarize available company news.",
    model=model,
    tools=[
        YFinanceTools(
            enable_company_news=True,
            enable_company_info=True,
        )
    ],
    instructions=[
        """Find and summarize available company news in simple language.
        If news is unavailable, continue using any other available company
        information. Never invent facts. Provide a useful answer with whatever
        information is available."""
    ],
    markdown=True,
)

# 2. Financial Analyst
quant_analyst = Agent(
    name="Financial Analyst",
    role="Analyze available stock and financial data.",
    model=model,
    tools=[
        YFinanceTools(
            enable_stock_price=True,
            enable_analyst_recommendations=True,
            enable_company_info=True,
            enable_stock_fundamentals=True,
            enable_key_financial_ratios=True,
        )
    ],
    instructions=[
        """Analyze available stock prices, financial statistics, and analyst
        recommendations. Explain the findings in simple language. If any data
        is missing or a tool fails, skip that item and continue with the data
        you have. Never invent numbers or claim unavailable data is current."""
    ],
    markdown=True,
)

# 3. Research Team Leader
research_team = Team(
    name="Stock Research Team",
    members=[news_researcher, quant_analyst],
    model=model,
    instructions=[
        """Create a clear stock report for everyday readers.
        Cover the overview, key numbers, recent news, growth opportunities, risks, and conclusion when available.
        Use simple language, concise bullets, and 180–220 words. Skip missing data.
        Give the most useful answer possible without inventing facts or guaranteeing returns."""
    ],
    markdown=True,
)

research_team.print_response(
    "Research NVIDIA (NVDA) using whatever company news and financial data "
    "is available. Explain the key findings, growth opportunities, and risks. "
    "If some information is unavailable, provide the rest of the report.",
    stream=True,
)


Output()

> ### Multi-Agent Collaboration:
> Look how cleanly the team orchestrated:
> 1. The Team Leader delegated web search to the **News Researcher**.
> 2. The Team Leader delegated financial metrics to the **Quantitative Analyst**.
> 3. Both specialists executed their tasks independently.
> 4. The Team Leader synthesized both streams into an executive Wall Street report.


## 7. Cloud-Managed Agent Harness — Anthropic

Unlike our local harness, Anthropic's Managed Agents run the agent loop in the cloud, reducing infrastructure management.

1. **Agent:** Define instructions and tools.
2. **Environment:** Set up cloud execution.
3. **Session:** Analyze portfolio data and generate an HTML dashboard.

*Note: Requires Anthropic API access and applicable billing. Skip if unavailable.*

In [ ]:
import anthropic

client = anthropic.Anthropic()

agent = client.beta.agents.create(
    name="Task Analyst",
    model="claude-haiku-5-5",
    system="Analyze tasks and summarize the results simply.",
    tools=[{"type": "agent_toolset_20260401"}],
)

environment = client.beta.environments.create(
    name="task-environment",
    config={"type": "cloud"},
)

print("Agent ready!")

Agent ready!


In [ ]:
# Run the research team and keep its complete report
response = research_team.run(
    "Research NVIDIA (NVDA): provide stock price, market cap, "
    "52-week range, financial metrics, analyst views, growth opportunities, "
    "recent news, and key risks. Use available data only.",
    stream=True,
)

report = response.content
print(report)

### Overview
NVIDIA Corporation is the world's leading semiconductor company powering the artificial intelligence (AI) revolution. It designs advanced graphics processing units (GPUs) and specialized computing platforms used across data centers, automotive, gaming, and robotics.

### Key Numbers
* **Stock Price:** $229.28
* **Market Cap:** $5.54 trillion
* **52-Week Range:** $164.27 – $243.37
* **Valuation:** 28.99 trailing P/E
* **Revenue Growth:** +105.9% year-over-year
* **Profit Margin:** 74.67% gross margin
* **Wall Street Consensus:** Strong Buy (Average price target: $328.72)

### Recent News & Updates
Direct real-time news articles were unavailable from automated feeds. However, NVIDIA's latest financial reports demonstrate massive triple-digit earnings growth, backed by strong liquidity with over $62 billion in cash reserves.

### Growth Opportunities
* **AI Data Centers:** Surging enterprise demand for accelerated computing hardware and AI superclusters.
* **Networking & Soft

In [ ]:
prompt = f"""
Create a professional, light-theme NVIDIA stock dashboard from this report:

{report}

Use 2-3 compact sections, clean financial charts, key metrics, and concise
growth drivers and risks. Keep text minimal and design polished, like a
professional equity research report. Use only verified figures; never invent data.

Save to /mnt/session/outputs/nvidia_research_dashboard.html.
"""

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
)

with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{
            "type": "user.message",
            "content": [{"type": "text", "text": prompt}],
        }],
    )

    for event in stream:
        if event.type == "agent.message":
            for item in event.content:
                if item.type == "text":
                    print(item.text)
        elif event.type == "session.status_idle":
            break

I built the dashboard as a single self-contained HTML file, saved to `/mnt/session/outputs/nvidia_research_dashboard.html`. It uses a light theme and has three compact sections:

1. **Valuation & Price Position:** a bar comparison of the 52-week low, current price, 52-week high and average price target, plus a marker showing where the stock sits in its 52-week range.
2. **Growth & Profitability:** revenue growth, a gross-margin donut chart, and the cash reserves and trailing P/E.
3. **Growth Drivers & Key Risks:** the three opportunities and three risks from the report, followed by the conclusion.

Every figure comes from your report. Three values are calculated from those figures, and the footer labels them as derived:
- Current price is 5.8% below the 52-week high.
- The average price target implies a 43.4% move from the current price.
- The current price sits at 82% of the 52-week range.

The report's "Profit Margin" label is shown as gross margin, since the report describes it that

In [ ]:
from IPython.display import HTML, display

files = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])

for f in files.data:
    if f.downloadable and f.filename.endswith(".html"):
        client.beta.files.download(f.id).write_to_file(f.filename)
        display(HTML(open(f.filename).read()))
        break

## 8. Which Harness Should You Pick? (The FDE Decision Matrix)

As a Forward Deployed Engineer (FDE), your role is to recommend and build the right harness architecture for your client's needs:

| Architecture | Who Runs the Loop? | Infrastructure Complexity | Best For |
|---|---|---|---|
| **Simple LLM Call** | Cloud API provider | None | Simple text summaries, drafting, creative generation (no real-time data or state) |
| **Framework Harness (Agno)** | You (in Colab, FastAPI, or Server) | Low (Pure Python) | Fast prototyping, internal enterprise agents, domain financial tools, complete code control |
| **Multi-Agent Teams** | You (via Team Coordinator) | Low to Medium | Complex multi-disciplinary workflows requiring separation of concerns (e.g. news research + quantitative modeling) |
| **Cloud Managed Agents (Claude)** | Anthropic | Zero (Serverless Cloud Sandbox) | Production client deployments needing sandboxed code execution, cloud bash, and zero server maintenance |

---

